# VLM: images and text

Image patches, a projector, and image-dependence checks.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

A VLM connects visual representations to text. A simple approach encodes image patches and projects them into a feature space. Real pretrained VLMs require their native processor, conversation format, and image-token conventions.

This tiny fusion classifier solves a synthetic task using image brightness and a text feature; it is not a generative VLM. VLM Studio supports its listed pretrained models and Research Lab architectures. Check SFT/GRPO and backend compatibility before training.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
images = torch.rand(64, 3, 8, 8)
instruction = torch.randint(0, 2, (64,))
# Shift brightness strongly enough that the task is visible in a tiny example.
images[:32] *= 0.2; images[32:] = 0.8 + images[32:] * 0.2
labels = ((images.mean((1, 2, 3)) > 0.5).long() ^ instruction)
class Fusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch = nn.Conv2d(3, 8, kernel_size=4, stride=4)
        self.text = nn.Embedding(2, 4)
        self.head = nn.Sequential(nn.Linear(12, 24), nn.ReLU(), nn.Linear(24, 2))
    def forward(self, image, text):
        visual = self.patch(image).mean((2, 3))
        return self.head(torch.cat([visual, self.text(text)], -1))
model = Fusion(); optimizer = torch.optim.AdamW(model.parameters(), lr=0.03)
for step in range(80):
    loss = F.cross_entropy(model(images, instruction), labels)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
with torch.no_grad():
    accuracy = (model(images, instruction).argmax(-1) == labels).float().mean()
    image_effect = (model(images, instruction) - model(torch.zeros_like(images), instruction)).abs().mean()
print('training accuracy:', accuracy.item(), 'image ablation delta:', image_effect.item())

## Try it yourself

Zero the image before training and compare accuracy and ablation delta. Create a held-out set: evaluating training data hides overfitting.

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [ViT](https://arxiv.org/abs/2010.11929)